# Problem 3 — Phase 3: Slack `C` by **tree depth**

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 2](hierarchical_problem3_phase2_truncated_svd.ipynb) · **Next:** [Phase 4](hierarchical_problem3_phase4_kernel_pilot.ipynb)

**Goal:** For each **branching-edge depth** (parent depth from `Root`: 0, 1, …), take **`EDGES_PER_DEPTH` = 3** trainable binary edges as a **local pilot**. For that depth only, loop **`C` in `C_GRID`**, fit those edges with a fresh router, and score **pooled / macro / pos-weighted F1, precision, and recall** (plus pooled accuracy) on the **test** split. Then pick the **best `C` per depth** (by `pooled_micro_f1` on the pilot edges at that depth).

**Full-tree** tuning (every edge × `C`) lives in [`archive/hierarchical_problem3_phase3_by_depth_full.ipynb`](archive/hierarchical_problem3_phase3_by_depth_full.ipynb).

In [10]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_classifier import MultilabelHierarchyRouter, svm_linear_binary_edge_factory
from hierarchical_summary_metrics import fit_binary_edges_subset, pooled_edge_f1_stats
from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import BinaryEdgeSpec, TopicTree, binary_edge_specs, load_topic_tree


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

EDGES_PER_DEPTH = 3  # pilot binary edges sampled per depth level
PILOT_SEED = 42
C_GRID = [1e-2, 0.1, 10.0, 20]

print("BASE", BASE)
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

BASE /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4
train 14465 test 3617


### Sample **3 edges per depth**

Keep specs where the **train** split has at least two classes. Group by **`spec.depth`**. At each depth, sample **`EDGES_PER_DEPTH`** edges (or fewer if not enough eligible). Reproducible via **`PILOT_SEED + depth`**. Writes **`problem3_phase3_pilot_edges_by_depth.json`**.

In [11]:
def eligible_specs_for_pilot(pool, tree: TopicTree, *, restrict: bool) -> list[BinaryEdgeSpec]:
    out: list[BinaryEdgeSpec] = []
    for spec in binary_edge_specs(tree):
        _, ytr = pool.binary_edge_dataset(
            spec.parent,
            spec.child,
            "train",
            restrict_to_parent_subtree=restrict,
        )
        if len(set(ytr)) >= 2:
            out.append(spec)
    return out


def sample_specs_per_depth(
    eligible: list[BinaryEdgeSpec],
    *,
    k_per_depth: int,
    seed: int,
) -> dict[int, list[BinaryEdgeSpec]]:
    """Up to ``k_per_depth`` specs per depth; RNG is ``Random(seed + depth)`` for reproducibility."""
    by_depth: dict[int, list[BinaryEdgeSpec]] = {}
    for spec in eligible:
        by_depth.setdefault(spec.depth, []).append(spec)
    out: dict[int, list[BinaryEdgeSpec]] = {}
    for d in sorted(by_depth):
        specs = by_depth[d]
        rng = random.Random(seed + d)
        kk = min(k_per_depth, len(specs))
        out[d] = rng.sample(specs, k=kk) if kk else []
    return out


eligible = eligible_specs_for_pilot(mldata, tree, restrict=RESTRICT)
pilot_by_depth = sample_specs_per_depth(
    eligible, k_per_depth=EDGES_PER_DEPTH, seed=PILOT_SEED
)

pilot_payload = {
    str(d): [{"parent": s.parent, "child": s.child, "depth": s.depth} for s in specs]
    for d, specs in sorted(pilot_by_depth.items())
}
pilot_path = BASE / "problem3_phase3_pilot_edges_by_depth.json"
with open(pilot_path, "w") as f:
    json.dump(pilot_payload, f, indent=2)

n_pilot = sum(len(v) for v in pilot_by_depth.values())
print(f"Eligible {len(eligible)}; pilot {n_pilot} edges across {len(pilot_by_depth)} depths -> {pilot_path}")
for d in sorted(pilot_by_depth):
    print(f"  depth {d}: {len(pilot_by_depth[d])} edges")

Eligible 101; pilot 12 edges across 4 depths -> /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase3_pilot_edges_by_depth.json
  depth 0: 3 edges
  depth 1: 3 edges
  depth 2: 3 edges
  depth 3: 3 edges


### Tune `C` **per depth**

For each **depth** with a non-empty pilot list: for each **`C` in `C_GRID`**, a **fresh** router; **[`fit_binary_edges_subset`](hierarchical_summary_metrics.py)** on **only that depth’s** edges; **[`pooled_edge_f1_stats`](hierarchical_summary_metrics.py)** on **`SPLIT`**. That yields **F1, precision, and recall** for **pooled micro**, **macro**, and **positive-weighted** aggregates, plus **pooled micro-accuracy**. Pick **best `C`** per depth by **`pooled_micro_f1`** only. Set **`RUN_PILOT = False`** to skip after the first run.

In [12]:
RUN_PILOT = True
VERBOSE_FIT = False  # set True to print each edge fit inside fit_binary_edges_subset

rows: list[dict] = []
if not any(pilot_by_depth.values()):
    print("No pilot edges; check eligibility / data.")
elif not RUN_PILOT:
    print("SKIP (RUN_PILOT=False)")
else:
    for depth in sorted(pilot_by_depth):
        specs_d = pilot_by_depth[depth]
        if not specs_d:
            continue
        pilot_tuples_d = [(s.parent, s.child) for s in specs_d]
        print("\n" + "=" * 60, flush=True)
        print(f"DEPTH {depth}  ({len(specs_d)} pilot edges)", flush=True)
        for C in C_GRID:
            fac = svm_linear_binary_edge_factory(
                max_features=MAX_FEATURES,
                text_vectorizer="tfidf",
                svm_kw=dict(C=float(C), dual=False, max_iter=8000),
            )
            router = MultilabelHierarchyRouter(tree, fac)
            t0 = time.perf_counter()
            fit_binary_edges_subset(
                router,
                mldata,
                specs_d,
                verbose=VERBOSE_FIT,
                restrict_to_parent_subtree=RESTRICT,
            )
            wall = time.perf_counter() - t0
            st = pooled_edge_f1_stats(
                router,
                mldata,
                pilot_tuples_d,
                SPLIT,
                restrict_to_parent_subtree=RESTRICT,
            )
            rows.append(
                {
                    "depth": depth,
                    "C": C,
                    "pooled_micro_f1": st["pooled_micro_f1"],
                    "pooled_micro_precision": st["pooled_micro_precision"],
                    "pooled_micro_recall": st["pooled_micro_recall"],
                    "pooled_micro_accuracy": st["pooled_micro_accuracy"],
                    "macro_f1": st["macro_f1"],
                    "macro_precision": st["macro_precision"],
                    "macro_recall": st["macro_recall"],
                    "pos_weighted_f1": st["pos_weighted_f1"],
                    "pos_weighted_precision": st["pos_weighted_precision"],
                    "pos_weighted_recall": st["pos_weighted_recall"],
                    "n_edges_scored": st["n_edges_used"],
                    "wall_time_sec": wall,
                }
            )
            print(
                f"  C={C:g}  pooled: F1={st['pooled_micro_f1']:.4f}  "
                f"P={st['pooled_micro_precision']:.4f}  R={st['pooled_micro_recall']:.4f}  |  "
                f"macro: F1={st['macro_f1']:.4f}  P={st['macro_precision']:.4f}  "
                f"R={st['macro_recall']:.4f}  |  pos_w: F1={st['pos_weighted_f1']:.4f}  "
                f"P={st['pos_weighted_precision']:.4f}  R={st['pos_weighted_recall']:.4f}  "
                f"({wall:.1f}s)",
                flush=True,
            )

    df = pd.DataFrame(rows)
    if not df.empty:
        display(df.sort_values(["depth", "C"]).round(4))
        out_csv = BASE / "problem3_phase3_pilot_c_by_depth_results.csv"
        df.to_csv(out_csv, index=False)
        print("Wrote", out_csv)

        best_rows = []
        for depth in sorted(df["depth"].unique()):
            sub = df[df["depth"] == depth]
            i = sub["pooled_micro_f1"].idxmax()
            r = sub.loc[i]
            best_rows.append(
                {
                    "depth": int(r["depth"]),
                    "best_C": r["C"],
                    "pooled_micro_f1": r["pooled_micro_f1"],
                    "pooled_micro_precision": r["pooled_micro_precision"],
                    "pooled_micro_recall": r["pooled_micro_recall"],
                    "pooled_micro_accuracy": r["pooled_micro_accuracy"],
                    "macro_f1": r["macro_f1"],
                    "macro_precision": r["macro_precision"],
                    "macro_recall": r["macro_recall"],
                    "pos_weighted_f1": r["pos_weighted_f1"],
                    "pos_weighted_precision": r["pos_weighted_precision"],
                    "pos_weighted_recall": r["pos_weighted_recall"],
                }
            )
        best_df = pd.DataFrame(best_rows)
        display(best_df.round(4))
        best_csv = BASE / "problem3_phase3_best_c_by_depth.csv"
        best_df.to_csv(best_csv, index=False)
        print("Wrote", best_csv)
        print("\nBest C per depth (chosen by pooled_micro_f1 on pilot edges); "
              "reported together: F1, precision P, recall R — pooled / macro / pos-weighted:")
        for _, br in best_df.iterrows():
            print(
                f"  depth {int(br['depth'])}: C={br['best_C']:g}\n"
                f"    pooled:  F1={br['pooled_micro_f1']:.4f}  P={br['pooled_micro_precision']:.4f}  "
                f"R={br['pooled_micro_recall']:.4f}  acc={br['pooled_micro_accuracy']:.4f}\n"
                f"    macro:   F1={br['macro_f1']:.4f}  P={br['macro_precision']:.4f}  "
                f"R={br['macro_recall']:.4f}\n"
                f"    pos_w:   F1={br['pos_weighted_f1']:.4f}  P={br['pos_weighted_precision']:.4f}  "
                f"R={br['pos_weighted_recall']:.4f}"
            )


DEPTH 0  (3 pilot edges)
  C=0.01  pooled: F1=0.7737  P=0.9231  R=0.6659  |  macro: F1=0.6859  P=0.9396  R=0.5767  |  pos_w: F1=0.7554  P=0.9298  R=0.6659  (15.6s)
  C=0.1  pooled: F1=0.8664  P=0.9116  R=0.8255  |  macro: F1=0.8462  P=0.9118  R=0.7931  |  pos_w: F1=0.8648  P=0.9120  R=0.8255  (12.2s)
  C=10  pooled: F1=0.8370  P=0.8372  R=0.8367  |  macro: F1=0.8272  P=0.8303  R=0.8243  |  pos_w: F1=0.8369  P=0.8372  R=0.8367  (13.1s)
  C=20  pooled: F1=0.8304  P=0.8278  R=0.8330  |  macro: F1=0.8197  P=0.8194  R=0.8201  |  pos_w: F1=0.8303  P=0.8278  R=0.8330  (19.5s)

DEPTH 1  (3 pilot edges)
  C=0.01  pooled: F1=0.3537  P=0.9280  R=0.2185  |  macro: F1=0.1304  P=0.3093  R=0.0826  |  pos_w: F1=0.3448  P=0.8179  R=0.2185  (4.8s)
  C=0.1  pooled: F1=0.7700  P=0.8605  R=0.6968  |  macro: F1=0.7737  P=0.9293  R=0.6656  |  pos_w: F1=0.7699  P=0.8626  R=0.6968  (8.1s)
  C=10  pooled: F1=0.7736  P=0.7536  R=0.7947  |  macro: F1=0.8438  P=0.8280  R=0.8606  |  pos_w: F1=0.7740  P=0.7545  R=0

,depth,C,pooled_micro_f1,pooled_micro_precision,pooled_micro_recall,pooled_micro_accuracy,macro_f1,macro_precision,macro_recall,pos_weighted_f1,pos_weighted_precision,pos_weighted_recall,n_edges_scored,wall_time_sec
0,0,0.01,0.7737,0.9231,0.6659,0.8848,0.6859,0.9396,0.5767,0.7554,0.9298,0.6659,3.0,15.5584
1,0,0.10,0.8664,0.9116,0.8255,0.9247,0.8462,0.9118,0.7931,0.8648,0.9120,0.8255,3.0,12.2422
2,0,10.00,0.8370,0.8372,0.8367,0.9036,0.8272,0.8303,0.8243,0.8369,0.8372,0.8367,3.0,13.1407
3,0,20.00,0.8304,0.8278,0.8330,0.8994,0.8197,0.8194,0.8201,0.8303,0.8278,0.8330,3.0,19.5351
4,1,0.01,0.3537,0.9280,0.2185,0.9012,0.1304,0.3093,0.0826,0.3448,0.8179,0.2185,3.0,4.8240
5,1,0.10,0.7700,0.8605,0.6968,0.9485,0.7737,0.9293,0.6656,0.7699,0.8626,0.6968,3.0,8.1088
6,1,10.00,0.7736,0.7536,0.7947,0.9424,0.8438,0.8280,0.8606,0.7740,0.7545,0.7947,3.0,7.6771
7,1,20.00,0.7657,0.7420,0.7910,0.9401,0.8409,0.8237,0.8592,0.7663,0.7433,0.7910,3.0,7.1362
8,2,0.01,0.7231,0.7861,0.6695,0.8563,0.5260,0.5687,0.5299,0.6429,0.6686,0.6695,3.0,0.9220
9,2,0.10,0.9176,0.9671,0.8729,0.9561,0.8663,0.9676,0.8090,0.9069,0.9670,0.8729,3.0,1.1486


Wrote /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase3_pilot_c_by_depth_results.csv


,depth,best_C,pooled_micro_f1,pooled_micro_precision,pooled_micro_recall,pooled_micro_accuracy,macro_f1,macro_precision,macro_recall,pos_weighted_f1,pos_weighted_precision,pos_weighted_recall
0,0,0.1,0.8664,0.9116,0.8255,0.9247,0.8462,0.9118,0.7931,0.8648,0.9120,0.8255
1,1,10.0,0.7736,0.7536,0.7947,0.9424,0.8438,0.8280,0.8606,0.7740,0.7545,0.7947
2,2,10.0,0.9227,0.9631,0.8856,0.9584,0.8902,0.9651,0.8393,0.9175,0.9636,0.8856
3,3,10.0,0.6994,0.7756,0.6368,0.8838,0.7025,0.7734,0.6470,0.6917,0.7629,0.6368


Wrote /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase3_best_c_by_depth.csv

Best C per depth (chosen by pooled_micro_f1 on pilot edges); reported together: F1, precision P, recall R — pooled / macro / pos-weighted:
  depth 0: C=0.1
    pooled:  F1=0.8664  P=0.9116  R=0.8255  acc=0.9247
    macro:   F1=0.8462  P=0.9118  R=0.7931
    pos_w:   F1=0.8648  P=0.9120  R=0.8255
  depth 1: C=10
    pooled:  F1=0.7736  P=0.7536  R=0.7947  acc=0.9424
    macro:   F1=0.8438  P=0.8280  R=0.8606
    pos_w:   F1=0.7740  P=0.7545  R=0.7947
  depth 2: C=10
    pooled:  F1=0.9227  P=0.9631  R=0.8856  acc=0.9584
    macro:   F1=0.8902  P=0.9651  R=0.8393
    pos_w:   F1=0.9175  P=0.9636  R=0.8856
  depth 3: C=10
    pooled:  F1=0.6994  P=0.7756  R=0.6368  acc=0.8838
    macro:   F1=0.7025  P=0.7734  R=0.6470
    pos_w:   F1=0.6917  P=0.7629  R=0.6368


### Final report

The **CSV** and **DataFrames** list, for each depth×`C` and for each **best** row, **F1**, **precision (P)**, **recall (R)**, and **pooled accuracy** together under three aggregations: **pooled micro** (one score on all pilot edge×test-document pairs), **macro** (mean over edges), and **pos_w** / **positive-weighted** (weighted by each edge’s positive count on the split). **Choosing `C`** still uses only **maximum pooled micro-F1** within `C_GRID`; the printed **best** block then shows **all** of these metrics at once so you can read precision/recall tradeoffs next to F1 for the selected slack.